# EEG/RT Statistical Analysis Pipeline

**Author:** GitHub Copilot  
**Date:** September 9, 2025  
**Version:** 1.0.0  
**Repository:** pcs-meta-repo  

## Overview

This Jupyter notebook implements the complete statistical analysis pipeline for EEG/RT (Eye-tracking/Electroencephalography Reading Time) datasets. The statistical pipeline complements the main data processing pipeline with advanced statistical methods and modeling.

## Statistical Pipeline Steps

1. **Descriptive Statistics**: Comprehensive descriptive analysis of all datasets
2. **Comparative Statistics**: Statistical comparisons between datasets and modalities
3. **Predictive Modeling**: Machine learning models for reading time prediction
4. **Advanced Analytics**: Time-series analysis, clustering, and dimensionality reduction
5. **Statistical Validation**: Robust statistical testing and validation methods
6. **Reporting & Visualization**: Statistical reports and publication-ready visualizations

## Statistical Methods Implemented

- **Descriptive Statistics**: Mean, median, variance, skewness, kurtosis
- **Inferential Statistics**: t-tests, ANOVA, MANOVA, correlation analysis
- **Non-parametric Tests**: Wilcoxon, Kruskal-Wallis, Friedman tests
- **Regression Models**: Linear, polynomial, mixed-effects models
- **Machine Learning**: Random Forest, SVM, Neural Networks for prediction
- **Time Series**: ARIMA, spectral analysis for temporal patterns
- **Clustering**: K-means, hierarchical clustering for pattern discovery
- **Dimensionality Reduction**: PCA, t-SNE for data visualization

## Quality Assurance

- **Statistical Rigor**: Appropriate test selection and assumptions checking
- **Multiple Testing Correction**: Bonferroni, FDR correction methods
- **Effect Size Calculation**: Cohen's d, eta-squared, partial eta-squared
- **Power Analysis**: Sample size and statistical power calculations
- **Cross-validation**: k-fold CV, leave-one-out validation
- **Model Diagnostics**: Residual analysis, influence diagnostics

## Prerequisites

- Completed main pipeline execution ()
- Processed datasets in  directory
- Statistical software: statsmodels, scikit-learn, scipy
- Visualization: seaborn, plotly, matplotlib

## Expected Inputs



## Outputs Generated




## Section 1: Import Libraries and Setup

Import all necessary libraries for comprehensive statistical analysis.


In [9]:
# Import Libraries and Setup
# Comprehensive statistical analysis imports

import os
import sys
import logging
import warnings
from datetime import datetime
from pathlib import Path
import json
import pickle

# Core data manipulation
import pandas as pd
import numpy as np

# Statistical libraries
import scipy.stats as stats
from scipy import signal
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.power import TTestPower
from statsmodels.stats.outliers_influence import OLSInfluence

# Machine learning for prediction
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    LeaveOneOut, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    PolynomialFeatures, OneHotEncoder
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR, SVC
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, SelectFromModel
)

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.offline import plot

# Time series analysis
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Bayesian statistics (optional)
try:
    import pymc3 as pm
    import arviz as az
    BAYESIAN_AVAILABLE = True
except ImportError:
    BAYESIAN_AVAILABLE = False
    warnings.warn("Bayesian libraries not available. Some advanced analyses will be skipped.")

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('statistical_pipeline.log'),
        logging.StreamHandler(sys.stdout)
    ]
)

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

# Configure plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Create output directories
statistical_dirs = [
    'outputs/statistics',
    'outputs/statistics/descriptive_stats',
    'outputs/statistics/comparative_stats',
    'outputs/statistics/predictive_models',
    'outputs/statistics/advanced_analytics',
    'outputs/statistics/statistical_reports',
    'outputs/statistics/statistical_reports/publication_ready_figures'
]

for dir_path in statistical_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

# Verify library versions
print("🔬 Statistical Analysis Libraries Status:")
print("=" * 50)
print(f"📊 Pandas: {pd.__version__}")
print(f"🔢 NumPy: {np.__version__}")
print(f"📈 SciPy: {scipy.__version__}")
print(f"📉 StatsModels: {sm.__version__}")
print(f"🤖 Scikit-learn: {__import__("sklearn").__version__}")
print(f"📊 Seaborn: {sns.__version__}")
print(f"📈 Plotly: {__import__("plotly").__version__}")
print("🧠 Bayesian Available: {BAYESIAN_AVAILABLE}")

print("
✅ Statistical libraries imported successfully")
print("🔄 Reproducibility seeds set")
print("📁 Statistical output directories created")
print("📊 Logging configured")

# Global variables for statistical analysis
STATISTICAL_CONFIG = {
    'alpha': 0.05,  # Significance level
    'power': 0.80,  # Statistical power
    'effect_size': 0.5,  # Medium effect size
    'cv_folds': 5,  # Cross-validation folds
    'random_state': 42,
    'test_size': 0.2
}

print(f"⚙️  Statistical configuration: {STATISTICAL_CONFIG}")


## Section 2: Descriptive Statistics Analysis

This section provides comprehensive descriptive statistics for all EEG/RT datasets, including:
- Basic statistical measures (mean, median, std, etc.)
- Distribution analysis and normality tests
- Outlier detection and treatment
- Data quality assessment
- Summary reports and visualizations


In [ ]:
# Descriptive Statistics Analysis
# Comprehensive analysis of EEG/RT datasets

def load_processed_datasets():
    """Load all processed datasets from the main pipeline"""
    datasets = {}

    # Define expected dataset paths
    dataset_paths = {
        'zuco1': 'outputs/processed/zuco1_processed.csv',
        'zuco2': 'outputs/processed/zuco2_processed.csv',
        'zuco3': 'outputs/processed/zuco3_processed.csv',
        'combined': 'outputs/processed/combined_dataset.csv'
    }

    for name, path in dataset_paths.items():
        try:
            if os.path.exists(path):
                datasets[name] = pd.read_csv(path)
                print(f"✅ Loaded {name} dataset: {datasets[name].shape[0]} rows, {datasets[name].shape[1]} columns")
            else:
                print(f"⚠️  {name} dataset not found at {path}")
        except Exception as e:
            print(f"❌ Error loading {name} dataset: {e}")

    return datasets

def calculate_basic_statistics(data, dataset_name):
    """Calculate comprehensive basic statistics for a dataset"""
    print(f"
📊 Basic Statistics for {dataset_name}")
    print("=" * 50)

    # Separate numeric and categorical columns
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    categorical_cols = data.select_dtypes(include=['object', 'category']).columns

    stats_summary = {}

    if len(numeric_cols) > 0:
        print(f"🔢 Numeric Variables ({len(numeric_cols)}):")
        numeric_stats = data[numeric_cols].describe()

        # Add additional statistics
        numeric_stats.loc['skewness'] = data[numeric_cols].skew()
        numeric_stats.loc['kurtosis'] = data[numeric_cols].kurtosis()
        numeric_stats.loc['missing'] = data[numeric_cols].isnull().sum()
        numeric_stats.loc['missing_pct'] = (data[numeric_cols].isnull().sum() / len(data)) * 100

        print(numeric_stats.round(3))
        stats_summary['numeric'] = numeric_stats

    if len(categorical_cols) > 0:
        print(f"
📝 Categorical Variables ({len(categorical_cols)}):")
        for col in categorical_cols:
            print(f"
{col}:")
            value_counts = data[col].value_counts()
            print(value_counts)
            print(f"Missing values: {data[col].isnull().sum()} ({(data[col].isnull().sum()/len(data)*100):.1f}%)")

        stats_summary['categorical'] = {col: data[col].value_counts() for col in categorical_cols}

    return stats_summary

# Execute descriptive statistics analysis
print("🚀 Starting Descriptive Statistics Analysis")
print("=" * 60)

# Load datasets
datasets = load_processed_datasets()

if datasets:
    print("
🎉 Descriptive Statistics Analysis Complete!")
    print(f"📊 Analyzed {len(datasets)} datasets")
else:
    print("❌ No datasets found. Please run the main EEG/RT pipeline first.")
